In [1]:
cd ..

/home/karaman/Documents/bet


In [2]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from src.prepare_data_inference import prepare_data_inference


In [3]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 11:10:07 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.1.179 instead (on interface ens33)
26/03/01 11:10:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 11:10:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

In [5]:
data_inference = prepare_data_inference('2024-06-20')

failed_withdrawals_30d
deposit_count_30d
withdrawal_count_30d
withdrawal_ratio
num_sessions_7d
net_game_result_7d
num_sessions_30d
avg_sessions_duration_30d
avg_bet_amount_30d
net_game_result_30d


In [6]:
silver_money_events.filter(F.col('event_ts')<='2024-06-10').filter(F.col('player_idx')==1437).orderBy(F.col('event_ts').desc()).show()

+--------------------+-------------------+----------+-------------+------------------+----------+
|            event_id|           event_ts|event_type|signed_amount| balance_after_txn|player_idx|
+--------------------+-------------------+----------+-------------+------------------+----------+
|a70576ae-5bc0-498...|2024-06-10 00:00:00|   session|         85.1| 5610.260000000001|      1437|
|5ca89474-8cfd-457...|2024-06-09 00:00:00|   session|        96.16| 5525.160000000001|      1437|
|f38b2b2b-5cc7-412...|2024-06-07 00:00:00|   session|        36.25| 5429.000000000001|      1437|
|656980e3-9c52-4e6...|2024-06-06 00:00:00|   session|        38.14| 5392.750000000001|      1437|
|8318e7d9-c364-40f...|2024-06-05 00:00:00|   session|        70.21| 5354.610000000001|      1437|
|4d09c5e8-8cd6-48f...|2024-05-30 00:00:00|   session|        99.77|           5259.64|      1437|
|c51e56f5-8d5f-4ac...|2024-05-30 00:00:00|   session|        24.76| 5284.400000000001|      1437|
|1e6256bc-e23f-4c4..

In [7]:
player_behavior.filter(F.col('reference_date')=='2024-06-20').filter(F.col('player_idx')==1437).show()

+----------+--------------+-----------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|reference_date|   balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+--------------+-----------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|      1437|    2024-06-20|5823.990000000002|4841.7300000000005|   610.1000000000001|   

In [8]:
data_inference.filter(F.col('player_idx')==1437).show()

+----------+--------------------+---------------------+-----------------+-----------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|  balance_30d_ago|   balance_7d_ago|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|
+----------+--------------------+---------------------+-----------------+-----------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|      1437|   610.1000000000001|              1532.35|4825.540000000001|5823.990000000002|                     0|                0|    

In [6]:
data_inference.count()

65407

In [7]:
player_behavior.filter(F.col('reference_date')=='2024-06-20').count()

65738

In [ ]:
ex_ids = player_behavior.filter(F.col('reference_date')=='2024-06-20').select('player_idx').join(data_inference.select('player_idx'), on='player_idx', how='left_anti')

In [14]:
player_behavior.filter(F.col('player_idx')==3468).show()

+----------+--------------+--------------+---------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|reference_date|balance_7d_ago|balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+--------------+--------------+---------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|      3468|    2024-06-20|        970.38|         970.38|                 0.0|                  0.0|     

In [13]:
ex_ids.show()

+----------+
|player_idx|
+----------+
|      3468|
|      9501|
|     35241|
|     65029|
|     93055|
|       547|
|      5883|
|     67535|
|     74071|
|     87862|
|     92649|
|     41257|
|     54513|
|     90113|
|     45497|
|     89682|
|     98154|
|      6232|
|     98634|
|     26639|
+----------+
only showing top 20 rows


In [ ]:
ex_frame = player_behavior.filter(F.col('reference_date')=='2024-06-20').join(ex_ids, on='player_idx', how='inner')

In [22]:
ex_frame.filter(F.col('deposit_count_30d')==1).show()

+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|reference_date|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|     72083|    2024-06-20|323.78000000000003|323.78000000000003|              113.09

In [17]:
from pyspark.sql.types import NumericType

In [20]:
# Identify numeric columns
numeric_cols = [
    field.name
    for field in ex_frame.schema.fields
    if isinstance(field.dataType, NumericType)
]

# Compute sums
df_sum = ex_frame.select([
    F.sum(F.col(c)).alias(c)
    for c in numeric_cols
])

df_sum.show()

+----------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|  20555979|237851.75999999995|237851.75999999995|              113.09|               113.09|              0|               0.0|  